# Alanine dipeptide: SBG versus full-covariance GMM + Proposition 3

This notebook compares the alanine-dipeptide experiment from *Scalable Equilibrium Sampling with Sequential Boltzmann Generators* against our basin-conditioned Method 2.

- **Paper baseline (SBG):** pretrained TarFlow proposal followed by the repository's continuous-time ULA-SMC implementation.
- **Method 2:** fit an unrestricted full-covariance GMM to the same standardized training trajectory, group its latent components by periodic Ramachandran basin, train basin-conditioned ECNF++ EGNN velocities using only within-basin flow pairs, and carry that fixed basin label through a Proposition-3 annealed sampler from the conditional GMM potential to the exact OpenMM target.

The target, trajectory split, force field, normalization, test reference, evaluator, particle count, annealing schedule, and resampling threshold come from the official Ace-A-Nme configuration. The GMM uses full covariance matrices; `reg_covar` is only a Cholesky-stability ridge. Cartesian configurations are centered and represented in the 63-dimensional center-of-mass-free subspace. Random rotation augmentation is deliberately not used because a fixed Cartesian GMM is not rotationally invariant.

Set `SMOKE_TEST=False` for the paper-scale comparison.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
!echo $CUDA_VISIBLE_DEVICES

In [ ]:
from __future__ import annotations

import math
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import torch
import torch.nn as nn
from dotenv import load_dotenv
from sklearn.mixture import GaussianMixture

load_dotenv(override=True)
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "transferable_samplers").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from the transferable-samplers-prop3 repository.")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from transferable_samplers.data.single_peptide_datamodule import SinglePeptideDataModule
from transferable_samplers.evaluation.evaluator import PeptideEnsembleEvaluator
from transferable_samplers.nn.egnn.egnn_dynamics_ad2_cat import EGNN_dynamics_AD2_cat
from transferable_samplers.utils.dataclasses import SamplesData
from transferable_samplers.utils.standardization import standardize_coords

SEED = 4201
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
SMOKE_TEST = True
RUN_OFFICIAL_SBG = False

NUM_COMPONENTS = 6
GMM_REG_COVAR = 1e-4
TRAIN_STEPS = 500 if SMOKE_TEST else 100_000
TRAIN_BATCH = 128 if SMOKE_TEST else 512
NUM_PARTICLES = 256 if SMOKE_TEST else 10_000
NUM_ANNEALING_STEPS = 20 if SMOKE_TEST else 100
EPSILON = 1e-5
ESS_THRESHOLD = 0.5
HUTCHINSON_SAMPLES = 1 if SMOKE_TEST else 4
DRIFT_BATCH = 32 if SMOKE_TEST else 128
ENERGY_BATCH = 64 if SMOKE_TEST else 256
LEARNING_RATE = 5e-4
SIGMA_MIN = 0.0

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_float32_matmul_precision("highest")
print({
    "root": str(REPO_ROOT),
    "device": str(DEVICE),
    "smoke_test": SMOKE_TEST,
    "hutchinson_samples": HUTCHINSON_SAMPLES,
})


## 1. Official SBG baseline

The command below runs the repository's unmodified `tarflow_Ace-A-Nme_ula` experiment. Its paper-scale defaults are 10,000 particles, 100 annealing steps, `init_eps=1e-5`, ESS threshold 0.5, multinomial resampling, target-energy cutoff 10, and 0.2% log-weight filtering. Results are saved separately and reused by the comparison cells.


In [ ]:
if "SCRATCH_DIR" not in os.environ:
    default_scratch = REPO_ROOT / "scratch"
    os.environ["SCRATCH_DIR"] = str(default_scratch)
    print(f"SCRATCH_DIR was not set; using ignored local cache: {default_scratch}")

ARTIFACT_ROOT = REPO_ROOT / "outputs" / "aldp_sbg_vs_prop3"
SBG_OUTPUT = ARTIFACT_ROOT / "official_sbg"
SBG_SAMPLE_FILE = SBG_OUTPUT / "test" / "Ace-A-Nme" / "samples_dict.pt"
SBG_DIAGNOSTICS_FILE = SBG_OUTPUT / "test" / "Ace-A-Nme" / "diagnostics.pt"

baseline_command = [
    "uv", "run", "python", "-m", "transferable_samplers.eval",
    "experiment=single_system/eval/tarflow_Ace-A-Nme_ula",
    "trainer=gpu" if torch.cuda.is_available() else "trainer=cpu",
    "logger=csv",
    f"paths.output_dir={SBG_OUTPUT}",
    f"callbacks.sampling_evaluation.output_dir={SBG_OUTPUT}",
]
print("Official SBG command:\n", " ".join(map(str, baseline_command)))
if RUN_OFFICIAL_SBG:
    SBG_OUTPUT.mkdir(parents=True, exist_ok=True)
    subprocess.run(baseline_command, cwd=REPO_ROOT, check=True)
elif not SBG_SAMPLE_FILE.exists():
    print("SBG artifact is absent. Set RUN_OFFICIAL_SBG=True to create it.")


## 2. Reuse the paper trajectory and OpenMM target

Ace-A-Nme is the paper's 300 K alanine-dipeptide system with 22 atoms. The datamodule downloads the official train/test splits and constructs the same OpenMM target and test reference used by the evaluator.


In [ ]:
scratch_root = Path(os.environ["SCRATCH_DIR"]) / "transferable-samplers"
datamodule = SinglePeptideDataModule(
    data_dir=str(scratch_root / "sequential-boltzmann-generators-data"),
    sequence="Ace-A-Nme",
    temperature=300,  # Keep this integer: the Hugging Face directory is Ace-A-Nme_300K.
    num_dimensions=3,
    num_atoms=22,
    batch_size=512,
    num_workers=0,
    num_eval_samples=10_000,
)
datamodule.prepare_data()
required_dataset_files = [
    datamodule.train_data_path,
    datamodule.val_data_path,
    datamodule.test_data_path,
    datamodule.pdb_path,
]
missing_dataset_files = [path for path in required_dataset_files if not Path(path).exists()]
if missing_dataset_files:
    raise FileNotFoundError(
        "Dataset download completed without the expected Ace-A-Nme_300K files:\n"
        + "\n".join(missing_dataset_files)
    )
train_raw = torch.from_numpy(np.load(datamodule.train_data_path)).float()
datamodule.std = (train_raw - train_raw.mean(dim=1, keepdim=True)).std()
eval_context = datamodule.prepare_eval(sequence="Ace-A-Nme", stage="test")
train_x = standardize_coords(train_raw, datamodule.std).to(DEVICE)

num_atoms, spatial_dim = train_x.shape[1:]
projector = torch.eye(num_atoms, dtype=torch.float64) - torch.ones(num_atoms, num_atoms, dtype=torch.float64) / num_atoms
eigenvalues, eigenvectors = torch.linalg.eigh(projector)
Q = torch.kron(eigenvectors[:, eigenvalues > 0.5], torch.eye(spatial_dim, dtype=torch.float64)).to(DEVICE, DTYPE)
DIM = Q.shape[1]

def x_to_y(x: torch.Tensor) -> torch.Tensor:
    return x.reshape(len(x), -1) @ Q

def y_to_x(y: torch.Tensor) -> torch.Tensor:
    return (y @ Q.T).reshape(len(y), num_atoms, spatial_dim)

train_y = x_to_y(train_x)
print({
    "train_samples": len(train_y),
    "mean_free_dimension": DIM,
    "normalization_std": float(datamodule.std),
    "test_reference": len(eval_context.true_data),
})


## 3. Fit the unrestricted full-covariance GMM

The component weights, means, and complete covariance matrices are fitted by maximum likelihood. Components are retained exactly as fitted and are not balanced manually. After the physical Ramachandran basins are identified below, these latent components are grouped by their overlap with those basins; the GMM parameters themselves are not refitted or reweighted.


In [ ]:
gmm_fit = GaussianMixture(
    n_components=NUM_COMPONENTS,
    covariance_type="full",
    reg_covar=GMM_REG_COVAR,
    n_init=5,
    max_iter=500,
    random_state=SEED,
).fit(train_y.detach().cpu().double().numpy())

gmm_weights = torch.as_tensor(gmm_fit.weights_, device=DEVICE, dtype=DTYPE)
gmm_means = torch.as_tensor(gmm_fit.means_, device=DEVICE, dtype=DTYPE)
gmm_covariances = torch.as_tensor(gmm_fit.covariances_, device=DEVICE, dtype=DTYPE)
gmm_cholesky = torch.linalg.cholesky(gmm_covariances)
gmm_precision = torch.cholesky_inverse(gmm_cholesky)
gmm_logdet = 2 * torch.log(torch.diagonal(gmm_cholesky, dim1=-2, dim2=-1)).sum(-1)
train_component = torch.as_tensor(gmm_fit.predict(train_y.detach().cpu().numpy()), device=DEVICE)
def sample_gmm(n: int, generator: torch.Generator | None = None, return_component: bool = False):
    component = torch.multinomial(gmm_weights, n, replacement=True, generator=generator)
    noise = torch.randn(n, DIM, device=DEVICE, generator=generator)
    y = gmm_means[component] + torch.bmm(gmm_cholesky[component], noise.unsqueeze(-1)).squeeze(-1)
    return (y, component) if return_component else y

def gmm_energy_score(y: torch.Tensor):
    delta = y[:, None, :] - gmm_means[None, :, :]
    precision_delta = torch.einsum("kij,bkj->bki", gmm_precision, delta)
    mahalanobis = torch.sum(delta * precision_delta, dim=-1)
    component_logp = (
        torch.log(gmm_weights)[None, :]
        - 0.5 * (DIM * math.log(2 * math.pi) + gmm_logdet[None, :] + mahalanobis)
    )
    responsibility = torch.softmax(component_logp, dim=1)
    score = torch.sum(responsibility[..., None] * (-precision_delta), dim=1)
    return -torch.logsumexp(component_logp, dim=1), score

print("fitted weights:", np.round(gmm_weights.cpu().numpy(), 5))
print("assigned counts:", torch.bincount(train_component, minlength=NUM_COMPONENTS).cpu().numpy())
print("covariance condition numbers:", np.round(torch.linalg.cond(gmm_covariances).cpu().numpy(), 2))


### Initial fitted-GMM Ramachandran density

This is the unweighted fitted-prior population before drift training or Proposition-3 transport. It uses the same seed as `run_prop3`, so these are exactly the particles that initialize the later Method-2 trajectory.


In [ ]:
def rama(samples: torch.Tensor, normalized: bool = True):
    xyz = samples.detach().cpu()
    if normalized:
        xyz = xyz * datamodule.std.cpu()
    xyz = xyz.numpy()
    trajectory = md.Trajectory(xyz, eval_context.topology)
    phi = md.compute_phi(trajectory)[1].reshape(len(samples), -1)[:, 0]
    psi = md.compute_psi(trajectory)[1].reshape(len(samples), -1)[:, 0]
    return np.rad2deg(phi), np.rad2deg(psi)

initial_gmm_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 2)
initial_gmm_y, initial_gmm_component = sample_gmm(
    NUM_PARTICLES, generator=initial_gmm_generator, return_component=True
)
initial_gmm_samples = y_to_x(initial_gmm_y).detach()
initial_gmm_phi, initial_gmm_psi = rama(initial_gmm_samples)
initial_component_mass = (
    torch.bincount(initial_gmm_component, minlength=NUM_COMPONENTS).float()
    / len(initial_gmm_component)
)
print("initial sampled component fractions:", initial_component_mass.cpu().numpy())

fig, ax = plt.subplots(figsize=(5.2, 4.4), constrained_layout=True)
density = ax.hexbin(
    initial_gmm_phi, initial_gmm_psi, gridsize=55, bins="log", mincnt=1
)
fig.colorbar(density, ax=ax, label="log count")
ax.set(
    xlim=(-180, 180), ylim=(-180, 180),
    xlabel=r"$\phi$ (deg)", ylabel=r"$\psi$ (deg)",
    title=f"Initial full-covariance GMM prior (K={NUM_COMPONENTS})",
)
plt.show()


### Initial GMM energy and interatomic-distance comparison

The fitted-prior particles are evaluated under the exact OpenMM target and compared with an equally sized subset of test MD conformations. Interatomic distances use the evaluator's definition: all unique upper-triangular atom-pair distances are pooled across conformations. The histograms use shared bins; energy values at or above 100 are collected in the final bin, matching the repository's plotting convention.


In [ ]:
from scipy.stats import wasserstein_distance

@torch.no_grad()
def batched_target_energy(samples: torch.Tensor, batch_size: int = ENERGY_BATCH):
    energies = []
    for start in range(0, len(samples), batch_size):
        energy = eval_context.target_energy.energy(
            samples[start : start + batch_size]
        )
        energies.append(energy.detach().cpu())
    return torch.cat(energies)

def pooled_interatomic_distances(samples: torch.Tensor):
    samples = samples.detach().cpu()
    num_atoms = samples.shape[1]
    upper = torch.triu(
        torch.ones(num_atoms, num_atoms, dtype=torch.bool), diagonal=1
    )
    return torch.cdist(samples, samples)[:, upper].reshape(-1)

comparison_count = min(len(initial_gmm_samples), len(eval_context.true_data.samples))
gmm_compare = initial_gmm_samples[:comparison_count]
md_compare = eval_context.true_data.samples[:comparison_count].detach().cpu()
gmm_prior_energy = batched_target_energy(gmm_compare)
md_energy = eval_context.true_data.E_target[:comparison_count].detach().cpu()
gmm_physical = gmm_compare.detach().cpu() * datamodule.std.cpu()
gmm_distances = pooled_interatomic_distances(gmm_physical)
md_distances = pooled_interatomic_distances(md_compare)

gmm_energy_finite = torch.isfinite(gmm_prior_energy)
md_energy_finite = torch.isfinite(md_energy)
if not bool(gmm_energy_finite.all() and md_energy_finite.all()):
    print({
        "finite_GMM_energy_fraction": float(gmm_energy_finite.float().mean()),
        "finite_MD_energy_fraction": float(md_energy_finite.float().mean()),
    })
gmm_energy_for_metrics = gmm_prior_energy[gmm_energy_finite]
md_energy_for_metrics = md_energy[md_energy_finite]

gmm_prior_comparison = {
    "num_conformations": comparison_count,
    "GMM_mean_target_energy": float(gmm_energy_for_metrics.mean()),
    "MD_mean_target_energy": float(md_energy_for_metrics.mean()),
    "GMM_median_target_energy": float(gmm_energy_for_metrics.median()),
    "MD_median_target_energy": float(md_energy_for_metrics.median()),
    "energy_W1": wasserstein_distance(
        md_energy_for_metrics.numpy(), gmm_energy_for_metrics.numpy()
    ),
    "GMM_fraction_energy_at_least_100": float((gmm_energy_for_metrics >= 100).float().mean()),
    "MD_fraction_energy_at_least_100": float((md_energy_for_metrics >= 100).float().mean()),
    "pooled_interatomic_distance_W1": wasserstein_distance(
        md_distances.numpy(), gmm_distances.numpy()
    ),
}
print(gmm_prior_comparison)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
energy_cap = 100.0
energy_min = float(torch.minimum(
    md_energy_for_metrics.min(), gmm_energy_for_metrics.min()
))
energy_bins = np.linspace(energy_min, energy_cap, 100)
for energy, label, color in [
    (md_energy_for_metrics, "test MD", "tab:green"),
    (gmm_energy_for_metrics, "fitted GMM prior", "tab:orange"),
]:
    axes[0].hist(
        torch.clamp(energy, max=energy_cap - 1e-4).numpy(),
        bins=energy_bins, density=True, histtype="step", lw=2, label=label, color=color
    )
axes[0].set(
    xlabel="exact target energy (last bin: >=100)", ylabel="density",
    title=f"Target energy; W1={gmm_prior_comparison['energy_W1']:.3g}"
)
axes[0].legend()

distance_min = float(torch.minimum(md_distances.min(), gmm_distances.min()))
distance_max = float(torch.maximum(md_distances.max(), gmm_distances.max()))
distance_bins = np.linspace(distance_min, distance_max, 100)
axes[1].hist(
    md_distances.numpy(), bins=distance_bins, density=True, histtype="step",
    lw=2, label="test MD", color="tab:green"
)
axes[1].hist(
    gmm_distances.numpy(), bins=distance_bins, density=True, histtype="step",
    lw=2, label="fitted GMM prior", color="tab:orange"
)
axes[1].set(
    xlabel="interatomic distance (nm)", ylabel="density",
    title=(
        "Pooled atom-pair distances; "
        f"W1={gmm_prior_comparison['pooled_interatomic_distance_W1']:.3g}"
    ),
)
axes[1].legend()
plt.show()


### Ramachandran-basin labels for mode-local transport

Physical modes are identified independently of the Cartesian GMM. The training angles are embedded periodically as $(\cos\phi,\sin\phi,\cos\psi,\sin\psi)$ and clustered into three basins. Each fitted GMM component is then assigned to the basin with which its training-data assignments overlap most, with a one-to-one coverage constraint so every basin owns at least one component. This component-to-basin map defines the fixed latent basin label used by both flow matching and Proposition 3.


In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans

NUM_BASINS = 3
BASIN_ANGLE_BATCH = 20_000
BASIN_KMEANS_SAMPLES = 50_000

def batched_phi_psi(samples: torch.Tensor, normalized: bool, batch_size: int):
    phi_parts, psi_parts = [], []
    for start in range(0, len(samples), batch_size):
        coordinates = samples[start : start + batch_size].detach().cpu()
        if normalized:
            coordinates = coordinates * datamodule.std.cpu()
        trajectory = md.Trajectory(coordinates.numpy(), eval_context.topology)
        phi = md.compute_phi(trajectory)[1].reshape(len(coordinates), -1)[:, 0]
        psi = md.compute_psi(trajectory)[1].reshape(len(coordinates), -1)[:, 0]
        phi_parts.append(phi)
        psi_parts.append(psi)
    return np.concatenate(phi_parts), np.concatenate(psi_parts)

train_phi, train_psi = batched_phi_psi(
    train_x, normalized=True, batch_size=BASIN_ANGLE_BATCH
)
train_torus_features = np.column_stack([
    np.cos(train_phi), np.sin(train_phi), np.cos(train_psi), np.sin(train_psi)
])
basin_rng = np.random.default_rng(SEED + 10)
basin_fit_indices = basin_rng.choice(
    len(train_torus_features),
    size=min(BASIN_KMEANS_SAMPLES, len(train_torus_features)),
    replace=False,
)
basin_kmeans = KMeans(
    n_clusters=NUM_BASINS, n_init=20, random_state=SEED + 10
).fit(train_torus_features[basin_fit_indices])
raw_basin_labels = basin_kmeans.predict(train_torus_features)

raw_basin_centers = []
for raw_label in range(NUM_BASINS):
    mask = raw_basin_labels == raw_label
    raw_basin_centers.append([
        np.arctan2(np.sin(train_phi[mask]).mean(), np.cos(train_phi[mask]).mean()),
        np.arctan2(np.sin(train_psi[mask]).mean(), np.cos(train_psi[mask]).mean()),
    ])
raw_basin_centers = np.asarray(raw_basin_centers)
# Stable display labels: increasing circular phi, then psi.
basin_order = np.lexsort((raw_basin_centers[:, 1], raw_basin_centers[:, 0]))
basin_label_remap = np.empty(NUM_BASINS, dtype=int)
basin_label_remap[basin_order] = np.arange(NUM_BASINS)
train_basin_labels_np = basin_label_remap[raw_basin_labels]
basin_centers = raw_basin_centers[basin_order]
basin_centers_degrees = np.rad2deg(basin_centers)
basin_counts = np.bincount(train_basin_labels_np, minlength=NUM_BASINS)
basin_weights = basin_counts / basin_counts.sum()
train_basin_labels = torch.as_tensor(
    train_basin_labels_np, dtype=torch.long, device=DEVICE
)
train_basin_index_pools = [
    torch.where(train_basin_labels == basin)[0] for basin in range(NUM_BASINS)
]

# Associate each global GMM component with one physical basin. The constrained
# seed assignment prevents a small basin from being left without a prior component.
component_basin_counts = torch.zeros(
    NUM_COMPONENTS, NUM_BASINS, dtype=torch.long, device=DEVICE
)
for component_index in range(NUM_COMPONENTS):
    mask = train_component == component_index
    component_basin_counts[component_index] = torch.bincount(
        train_basin_labels[mask], minlength=NUM_BASINS
    )
component_to_basin = component_basin_counts.argmax(dim=1)
basin_rows, component_columns = linear_sum_assignment(
    -component_basin_counts.T.detach().cpu().numpy()
)
for basin_index, component_index in zip(basin_rows, component_columns):
    component_to_basin[component_index] = int(basin_index)
basin_component_indices = [
    torch.where(component_to_basin == basin)[0] for basin in range(NUM_BASINS)
]
basin_prior_mass = torch.stack([
    gmm_weights[indices].sum() for indices in basin_component_indices
])
if any(len(indices) == 0 for indices in basin_component_indices):
    raise RuntimeError("Every Ramachandran basin must own at least one GMM component.")

def gmm_conditional_energy_score(y: torch.Tensor, basin: torch.Tensor):
    """Energy and score of q0(y | basin), excluding the shared basin mass."""
    energy = torch.empty(len(y), device=y.device, dtype=y.dtype)
    score = torch.empty_like(y)
    for basin_index, indices in enumerate(basin_component_indices):
        mask = basin == basin_index
        if not bool(mask.any()):
            continue
        delta = y[mask, None, :] - gmm_means[indices][None, :, :]
        precision_delta = torch.einsum(
            "kij,bkj->bki", gmm_precision[indices], delta
        )
        mahalanobis = torch.sum(delta * precision_delta, dim=-1)
        conditional_weights = gmm_weights[indices] / basin_prior_mass[basin_index]
        component_logp = (
            torch.log(conditional_weights)[None, :]
            - 0.5 * (
                DIM * math.log(2 * math.pi)
                + gmm_logdet[indices][None, :]
                + mahalanobis
            )
        )
        responsibility = torch.softmax(component_logp, dim=1)
        energy[mask] = -torch.logsumexp(component_logp, dim=1)
        score[mask] = torch.sum(
            responsibility[..., None] * (-precision_delta), dim=1
        )
    return energy, score

print("basin | center phi | center psi | count | empirical weight")
for basin in range(NUM_BASINS):
    print(
        f"{basin:5d} | {basin_centers_degrees[basin, 0]:10.2f} | "
        f"{basin_centers_degrees[basin, 1]:10.2f} | {basin_counts[basin]:5d} | "
        f"{basin_weights[basin]:.5f}"
    )
print("component -> basin:", component_to_basin.detach().cpu().tolist())
print("GMM prior basin masses:", np.round(basin_prior_mass.detach().cpu().numpy(), 5))

plot_count = min(60_000, len(train_phi))
plot_indices = basin_rng.choice(len(train_phi), size=plot_count, replace=False)
colors = np.asarray(["tab:blue", "tab:orange", "tab:green"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
for basin in range(NUM_BASINS):
    mask = train_basin_labels_np[plot_indices] == basin
    selected = plot_indices[mask]
    axes[0].scatter(
        np.rad2deg(train_phi[selected]), np.rad2deg(train_psi[selected]),
        s=3, alpha=0.18, color=colors[basin], label=f"basin {basin}"
    )
axes[0].scatter(
    basin_centers_degrees[:, 0], basin_centers_degrees[:, 1],
    c=colors, marker="x", s=100, linewidths=3
)
axes[0].set(
    xlim=(-180, 180), ylim=(-180, 180),
    xlabel=r"$\phi$ (deg)", ylabel=r"$\psi$ (deg)",
    title="Periodic training-data basin assignments",
)
axes[0].legend(markerscale=3)
axes[1].bar(np.arange(NUM_BASINS), basin_weights, color=colors)
axes[1].set(
    xticks=np.arange(NUM_BASINS), xlabel="basin", ylabel="training fraction",
    ylim=(0, max(0.55, 1.1 * basin_weights.max())),
    title="Empirical basin weights (not forced uniform)",
)
plt.show()


## 4. Train the Ramachandran-basin-conditioned ECNF++ drift

This uses one copy of the paper repository's equivariant EGNN architecture per Ramachandran basin (width 256 and depth 5). Each network acts in 66 Cartesian coordinates, while the wrapper projects its mean-free velocity into the 63-dimensional GMM coordinates. A prior sample is paired only with a biased-data conformation carrying the same fixed basin label, so the learned drift cannot create cross-basin training pairs.


In [ ]:
class BasinConditionedProjectedEGNNVelocity(nn.Module):
    def __init__(self):
        super().__init__()
        self.egnn_by_basin = nn.ModuleList([
            EGNN_dynamics_AD2_cat(
                num_atoms=num_atoms,
                num_dimensions=spatial_dim,
                channels=256,
                num_layers=5,
            )
            for _ in range(NUM_BASINS)
        ])
        self.register_buffer("basis", Q)

    def forward(self, y: torch.Tensor, t: torch.Tensor, basin: torch.Tensor):
        x_flat = y @ self.basis.T
        velocity_y = torch.zeros_like(y)
        for basin_index, basin_egnn in enumerate(self.egnn_by_basin):
            mask = basin == basin_index
            if not bool(mask.any()):
                continue
            basin_t = t if t.ndim == 0 else t[mask]
            velocity_x = basin_egnn(basin_t, x_flat[mask])
            velocity_y[mask] = velocity_x @ self.basis
        return velocity_y

drift = BasinConditionedProjectedEGNNVelocity().to(DEVICE)
optimizer = torch.optim.AdamW(drift.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
generator = torch.Generator(device=DEVICE).manual_seed(SEED + 1)
loss_history = []
drift.train()

for step in range(TRAIN_STEPS):
    y0, component = sample_gmm(TRAIN_BATCH, generator=generator, return_component=True)
    basin = component_to_basin[component]
    y1 = torch.empty_like(y0)
    for basin_index, pool in enumerate(train_basin_index_pools):
        mask = basin == basin_index
        count = int(mask.sum())
        if count:
            chosen = pool[torch.randint(len(pool), (count,), device=DEVICE, generator=generator)]
            y1[mask] = train_y[chosen]

    t = torch.rand(TRAIN_BATCH, 1, device=DEVICE, generator=generator)
    yt = (1 - (1 - SIGMA_MIN) * t) * y0 + t * y1
    target_velocity = y1 - (1 - SIGMA_MIN) * y0
    loss = (drift(yt, t[:, 0], basin) - target_velocity).square().mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(drift.parameters(), 10.0)
    optimizer.step()
    loss_history.append(float(loss.detach()))
    if (step + 1) % max(100, TRAIN_STEPS // 10) == 0:
        print(f"{step + 1}/{TRAIN_STEPS}: loss={loss_history[-1]:.6f}")

drift.eval()
plt.figure(figsize=(6, 3))
plt.plot(loss_history)
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("flow-matching MSE")
plt.title("Ramachandran-basin-conditioned EGNN drift")
plt.show()


## 5. Proposition-3 SMC

For (U_t=(1-t)U_0+tU_1), the dynamics and incremental log weight are

[
dX_t = [-\epsilon\nabla U_t(X_t)+v_t(X_t)]dt + \sqrt{2\epsilon},dW_t,
]

[
d\log w_t = [\nabla\!\cdot v_t-\nabla U_t\!\cdot v_t+U_0-U_1]dt.
]

The sampler runs on the augmented state $(X_t,Z)$, where the Ramachandran-basin label $Z$ remains fixed along each trajectory and selects both the conditional GMM potential and the basin-specific drift. The prior basin mass $\rho_Z$ is included in both endpoints, $q_0(y,z)=\rho_z q_0(y\mid z)$ and $q_1(y,z)=\rho_z p(y)$, so it cancels from the incremental weight and the marginal terminal target remains the exact physical target. The divergence is estimated with Hutchinson probes. Multinomial resampling uses the same ESS threshold as the SBG configuration and resamples the basin label together with its particle.


In [ ]:
@torch.no_grad()
def target_energy_score(y: torch.Tensor):
    energies, scores = [], []
    for start in range(0, len(y), ENERGY_BATCH):
        y_batch = y[start : start + ENERGY_BATCH]
        x_batch = y_to_x(y_batch)
        energy, gradient_x = eval_context.target_energy.energy_and_grad(x_batch)
        energies.append(energy.to(DEVICE))
        scores.append(-(gradient_x.to(DEVICE).reshape(len(y_batch), -1) @ Q))
    return torch.cat(energies), torch.cat(scores)

def drift_and_divergence(y: torch.Tensor, t: torch.Tensor, basin: torch.Tensor):
    velocity_parts, divergence_parts = [], []
    for start in range(0, len(y), DRIFT_BATCH):
        leaf = y[start : start + DRIFT_BATCH].detach().requires_grad_(True)
        basin_batch = basin[start : start + DRIFT_BATCH]
        with torch.enable_grad():
            velocity = drift(leaf, t, basin_batch)
            divergence = torch.zeros(len(leaf), device=DEVICE)
            for _ in range(HUTCHINSON_SAMPLES):
                probe = torch.empty_like(leaf).bernoulli_(0.5).mul_(2).sub_(1)
                vector_jacobian = torch.autograd.grad(
                    (velocity * probe).sum(), leaf, retain_graph=True
                )[0]
                divergence += (vector_jacobian * probe).sum(1) / HUTCHINSON_SAMPLES
        velocity_parts.append(velocity.detach())
        divergence_parts.append(divergence.detach())
    return torch.cat(velocity_parts), torch.cat(divergence_parts)

def normalized_ess(logw: torch.Tensor) -> float:
    weights = torch.softmax(logw, dim=0)
    return float(1 / (len(weights) * weights.square().sum()))

def run_prop3(resample: bool = True, seed: int = SEED + 2):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    y, ancestry = sample_gmm(NUM_PARTICLES, generator=generator, return_component=True)
    basin = component_to_basin[ancestry]
    logw = torch.zeros(NUM_PARTICLES, device=DEVICE)
    dt = 1.0 / NUM_ANNEALING_STEPS
    diagnostics = {
        "t": [], "ess": [], "resampled": [],
        "raw_component_mass": [], "raw_basin_mass": [],
        "weighted_basin_mass": [],
    }
    num_resamples = 0

    for step in range(NUM_ANNEALING_STEPS):
        t = torch.tensor((step + 0.5) * dt, device=DEVICE)
        U0, score0 = gmm_conditional_energy_score(y, basin)
        U1, score1 = target_energy_score(y)
        velocity, divergence = drift_and_divergence(y, t, basin)
        grad_Ut = -((1 - t) * score0 + t * score1)

        logw += (divergence - (grad_Ut * velocity).sum(1) + U0 - U1) * dt
        y += (-EPSILON * grad_Ut + velocity) * dt
        y += math.sqrt(2 * EPSILON * dt) * torch.randn(y.shape, device=DEVICE, generator=generator)

        current_ess = normalized_ess(logw)
        normalized_weights = torch.softmax(logw, dim=0)
        weighted_basin_mass = torch.stack([
            normalized_weights[basin == basin_index].sum()
            for basin_index in range(NUM_BASINS)
        ])
        did_resample = bool(resample and current_ess < ESS_THRESHOLD)
        if did_resample:
            index = torch.multinomial(torch.softmax(logw, 0), len(logw), replacement=True, generator=generator)
            y = y[index]
            ancestry = ancestry[index]
            basin = basin[index]
            logw.zero_()
            num_resamples += 1

        diagnostics["t"].append(float((step + 1) * dt))
        diagnostics["ess"].append(current_ess)
        diagnostics["resampled"].append(did_resample)
        diagnostics["raw_component_mass"].append(
            (torch.bincount(ancestry, minlength=NUM_COMPONENTS).float() / len(ancestry)).cpu()
        )
        diagnostics["raw_basin_mass"].append(
            (torch.bincount(basin, minlength=NUM_BASINS).float() / len(basin)).cpu()
        )
        diagnostics["weighted_basin_mass"].append(weighted_basin_mass.cpu())
        if not torch.isfinite(y).all() or not torch.isfinite(logw).all():
            raise FloatingPointError(f"Non-finite Proposition-3 state at step {step + 1}")
        print(
            f"step {step + 1:3d}/{NUM_ANNEALING_STEPS}: "
            f"ESS/N={current_ess:.4f}, resampled={did_resample}"
        )

    target_energy, _ = target_energy_score(y)
    return {
        "y": y.detach(),
        "samples": y_to_x(y).detach(),
        "target_energy": target_energy.detach(),
        "logw": logw.detach(),
        "basin": basin.detach(),
        "component_ancestry": ancestry.detach(),
        "diagnostics": diagnostics,
        "num_resamples": num_resamples,
    }


In [ ]:
method2 = run_prop3(resample=True)
print({
    "final_segment_ESS/N": normalized_ess(method2["logw"]),
    "resampling_events": method2["num_resamples"],
    "particles": len(method2["samples"]),
})


## 6. Matched evaluation

The repository evaluator computes the same energy Wasserstein, Ramachandran-torus Wasserstein, TICA Wasserstein, and clustering metrics used by the paper. If the official SBG artifact exists, both methods are evaluated together against the same test reference.


In [ ]:
def samples_data_to_cpu(data: SamplesData) -> SamplesData:
    return SamplesData(
        samples=data.samples.detach().cpu(),
        E_target=data.E_target.detach().cpu(),
        logw=data.logw.detach().cpu() if data.logw is not None else None,
    )

comparison_samples = {
    "gmm_prop3": SamplesData(
        method2["samples"].cpu(),
        method2["target_energy"].cpu(),
        logw=method2["logw"].cpu(),
    )
}
sbg_diagnostics = None
if SBG_SAMPLE_FILE.exists():
    official = torch.load(SBG_SAMPLE_FILE, map_location="cpu", weights_only=False)
    comparison_samples["sbg_smc"] = samples_data_to_cpu(official["smc"])
    if SBG_DIAGNOSTICS_FILE.exists():
        sbg_diagnostics = torch.load(SBG_DIAGNOSTICS_FILE, map_location="cpu", weights_only=False)
else:
    print("Official SBG artifact not found; evaluating Method 2 only.")

evaluator = PeptideEnsembleEvaluator(
    fix_symmetry=True,
    drop_unfixable_symmetry=False,
    num_eval_samples=min(10_000, NUM_PARTICLES),
    do_plots=False,
)
metrics = evaluator.evaluate(comparison_samples, eval_context, prefix="test/Ace-A-Nme")
for key, value in sorted(metrics.items()):
    if any(token in key for token in ("wasserstein", "effective-sample-size", "median-energy", "mean-energy", "jsd")):
        scalar = float(value) if isinstance(value, (float, int, torch.Tensor)) else value
        print(f"{key}: {scalar}")


## 7. Ramachandran, ESS, and basin-mass diagnostics

The SBG SMC output is resampled at the endpoint, so its output weights are uniform. Its pre-resampling ESS trajectory is read from the saved diagnostics. Method-2 ESS is the segment ESS between resampling events; resampling times are marked explicitly. For Method 2, the raw particle fraction after any resampling and the weighted fraction immediately before resampling are shown separately for every fixed basin label.


In [ ]:
plot_sets = [("test MD", eval_context.true_data.samples, None, False)]
if "sbg_smc" in comparison_samples:
    plot_sets.append(("official SBG SMC", comparison_samples["sbg_smc"].samples, None, True))
plot_sets.append(("basin-conditioned GMM + Prop. 3", comparison_samples["gmm_prop3"].samples, comparison_samples["gmm_prop3"].logw, True))

fig, axes = plt.subplots(1, len(plot_sets), figsize=(5 * len(plot_sets), 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (title, samples, logw, normalized) in zip(axes, plot_sets):
    phi, psi = rama(samples, normalized=normalized)
    if logw is None:
        ax.hexbin(phi, psi, gridsize=55, bins="log", mincnt=1)
    else:
        weights = torch.softmax(logw, 0).numpy()
        ax.hexbin(phi, psi, C=weights, reduce_C_function=np.sum, gridsize=55, mincnt=1)
    ax.set(xlim=(-180, 180), ylim=(-180, 180), xlabel=r"$\phi$ (deg)", ylabel=r"$\psi$ (deg)", title=title)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
times = np.asarray(method2["diagnostics"]["t"])
method2_ess = np.asarray(method2["diagnostics"]["ess"])
ax.plot(times, method2_ess, label="Method 2: Prop. 3")
for time, flag in zip(times, method2["diagnostics"]["resampled"]):
    if flag:
        ax.axvline(time, color="C0", alpha=0.25, linestyle="--")
if sbg_diagnostics is not None:
    sbg_diag = sbg_diagnostics["diagnostics"]
    ax.plot(np.asarray(sbg_diag["t"], float), np.asarray(sbg_diag["ess"], float), label="Official SBG SMC")
ax.axhline(ESS_THRESHOLD, color="black", linestyle=":", label="resampling threshold")
ax.set(xlabel="annealing time", ylabel="ESS/N", ylim=(0, 1), title="Annealing weight efficiency")
ax.legend()
plt.show()

raw_basin_mass = np.stack(method2["diagnostics"]["raw_basin_mass"])
weighted_basin_mass = np.stack(method2["diagnostics"]["weighted_basin_mass"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for basin_index in range(NUM_BASINS):
    axes[0].plot(times, raw_basin_mass[:, basin_index], label=f"basin {basin_index}")
    axes[1].plot(times, weighted_basin_mass[:, basin_index], label=f"basin {basin_index}")
axes[0].set(
    xlabel="annealing time", ylabel="particle fraction", ylim=(0, 1),
    title="Raw basin ancestry after resampling",
)
axes[1].set(
    xlabel="annealing time", ylabel="weighted fraction", ylim=(0, 1),
    title="Weighted basin mass before resampling",
)
for ax in axes:
    ax.legend()
plt.show()


## Interpretation checklist

Use the comparison for three separate questions:

1. **Proposal/transport quality:** compare raw Ramachandran and target-energy distributions before final weighting or resampling.
2. **Correction efficiency:** compare ESS trajectories and the number/timing of resampling events. Do not interpret the final uniform SMC weights as an ESS of the original trajectories.
3. **Final equilibrium quality:** compare energy, torus, and TICA Wasserstein metrics after resampling.

For a paper-matched run, use `SMOKE_TEST=False`, run the official baseline once with `RUN_OFFICIAL_SBG=True`, and repeat both methods over seeds 0, 1, and 2. The reported SBG baseline applies energy and log-weight filtering; Method 2 is left unfiltered by default so its Proposition-3 correction remains transparent. If filtering is introduced, report filtered and unfiltered results separately.
